In [1]:
import pandas as pd
import os

Merging NMT datasets

In [ ]:
# ── Column mappings per year ──────────────────────────────────────────────────
# Target columns in the merged dataset:
#   outid, RegName, RegTypeName, PTRegName, SexTypeName, TerTypeName,
#   MathBall100, Birth, TestDate, Year

COMMON_COLS = ["outid", "RegName", "RegTypeName", "PTRegName",
               "SexTypeName", "TerTypeName", "Birth", "TestDate"]

# Math score column differs by year
MATH_COL_MAP = {
    2021: "MathBall100",
    2022: "Block3Ball100",
    2023: "MathBlockBall100",
    2024: "MathBlockBall100",
    2025: "MathBlockBall100",
}

FILE_MAP = {
    2021: "data/Odata2021File.csv",
    2022: "data/Odata2022File.csv",
    2023: "data/Odata2023File.csv",
    2024: "data/Odata2024File.csv",
    2025: "data/Odata2025File.csv",
}

READ_KWARGS = dict(
    sep=";",
    quotechar='"',
    engine="python",
    on_bad_lines="skip",
    encoding="utf-8-sig"
)

# ── Load, filter, rename, collect ─────────────────────────────────────────────
frames = []

for year, filepath in FILE_MAP.items():
    if not os.path.exists(filepath):
        print(f"⚠️  File not found, skipping: {filepath}")
        continue

    print(f"📂 Loading {year}...")
    df = pd.read_csv(filepath, **READ_KWARGS)

    print(f"   Raw shape: {df.shape}")
    print(f"   Columns: {list(df.columns)}")

    # Filter out graduates from previous years
    if "RegTypeName" in df.columns:
        before = len(df)
        df = df[df["RegTypeName"] != "Випускник минулих років"]
        print(f"   Dropped {before - len(df)} 'Випускник минулих років' rows")
    else:
        print(f"   ⚠️  'RegTypeName' not found — skipping filter for {year}")

    # Identify math score column for this year
    math_col = MATH_COL_MAP[year]
    if math_col not in df.columns:
        # Fallback: try to find any column containing "math" or "Ball100" (case-insensitive)
        candidates = [c for c in df.columns
                      if "ball100" in c.lower() or "math" in c.lower()]
        if candidates:
            math_col = candidates[0]
            print(f"   ℹ️  Math col not found as '{MATH_COL_MAP[year]}', "
                  f"using fallback: '{math_col}'")
        else:
            print(f"   ❌ No math score column found for {year}, skipping.")
            continue

    # Select and rename math column → unified "MathBall100"
    cols_to_keep = [c for c in COMMON_COLS if c in df.columns] + [math_col]
    missing = [c for c in COMMON_COLS if c not in df.columns]
    if missing:
        print(f"   ⚠️  Missing common columns for {year}: {missing}")

    df = df[cols_to_keep].copy()
    df = df.rename(columns={math_col: "MathBall100"})

    # Add year indicator
    df["Year"] = year

    # Convert math score to numeric (some years use comma as decimal separator)
    df["MathBall100"] = (
        df["MathBall100"]
        .astype(str)
        .str.replace(",", ".", regex=False)
    )
    df["MathBall100"] = pd.to_numeric(df["MathBall100"], errors="coerce")

    print(f"   ✅ Kept {len(df):,} rows after filter")
    frames.append(df)

# ── Merge all years ───────────────────────────────────────────────────────────
if not frames:
    raise RuntimeError("No data loaded — check file paths and column names.")

merged = pd.concat(frames, ignore_index=True)

# Ensure consistent column order
final_cols = ["outid", "RegName", "RegTypeName", "PTRegName",
              "SexTypeName", "TerTypeName", "MathBall100",
              "Birth", "TestDate", "Year"]
merged = merged[[c for c in final_cols if c in merged.columns]]

print(f"\n📊 Final merged shape: {merged.shape}")
print(merged.dtypes)
print(merged.head())

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = "data/nmt_merged2021-2025.csv"
merged.to_csv(out_path, index=False, encoding="utf-8")
print(f"\n💾 Saved to: {out_path}")

📂 Loading 2021...
   Raw shape: (240946, 147)
   Columns: ['outid', 'Birth', 'SexTypeName', 'RegName', 'AREANAME', 'TERNAME', 'RegTypeName', 'TerTypeName', 'ClassProfileNAME', 'ClassLangName', 'EONAME', 'EOTypeName', 'EORegName', 'EOAreaName', 'EOTerName', 'EOParent', 'UMLTest', 'UMLTestStatus', 'UMLBall100', 'UMLBall12', 'UMLBall', 'UMLAdaptScale', 'UMLPTName', 'UMLPTRegName', 'UMLPTAreaName', 'UMLPTTerName', 'UkrTest', 'UkrSubTest', 'UkrTestStatus', 'UkrBall100', 'UkrBall12', 'UkrBall', 'UkrAdaptScale', 'UkrPTName', 'UkrPTRegName', 'UkrPTAreaName', 'UkrPTTerName', 'HistTest', 'HistLang', 'HistTestStatus', 'HistBall100', 'HistBall12', 'HistBall', 'HistPTName', 'HistPTRegName', 'HistPTAreaName', 'HistPTTerName', 'MathTest', 'MathLang', 'MathTestStatus', 'MathBall100', 'MathBall12', 'MathDpaLevel', 'MathBall', 'MathPTName', 'MathPTRegName', 'MathPTAreaName', 'MathPTTerName', 'MathStTest', 'MathStLang', 'MathStTestStatus', 'MathStBall12', 'MathStBall', 'MathStPTName', 'MathStPTRegName', 

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd1 in position 4072: invalid continuation byte

In [ ]:

# ── Column mappings per year ───────────────────────────────────────────────────────
# Unified target columns in merged dataset:
#   OUTID, RegName, RegTypeName, PTRegName, SexTypeName, TerTypeName,
#   MathBall100, Birth, TestDate, Year

# Math score column name differs by year
MATH_COL_MAP = {
    2021: "MathBall100",
    2022: "Block3Ball100",
    2023: "MathBlockBall100",
    2024: "MathBlockBall100",
    2025: "MathBlockBall100",
}

# PTRegName column name differs in 2021
PTREG_COL_MAP = {
    2021: "MathPTRegName",
    2022: "PTRegName",
    2023: "PTRegName",
    2024: "PTRegName",
    2025: "PTRegName",
}

FILE_MAP = {
    2021: "data/Odata2021File.csv",
    2022: "data/Odata2022File.csv",
    2023: "data/Odata2023File.csv",
    2024: "data/Odata2024File.csv",
    2025: "data/Odata2025File.csv",
}

# Columns that have the same name across all years
STABLE_COLS = ["outid", "RegName", "RegTypeName",
               "SexTypeName", "TerTypeName", "Birth", "TestDate"]

READ_KWARGS = dict(
    sep=";",
    quotechar='"',
    engine="python",
    on_bad_lines="skip",
    encoding="utf-8"
)

# ── Load, filter, rename, collect ─────────────────────────────────────────────
frames = []

for year, filepath in FILE_MAP.items():
    if not os.path.exists(filepath):
        print(f"⚠️  File not found, skipping: {filepath}")
        continue

    print(f"\n📂 Loading {year}...")
    df = pd.read_csv(filepath, **READ_KWARGS)
    print(f"   Raw shape: {df.shape}")

    # ── Filter out previous-year graduates ────────────────────────────────────
    if "RegTypeName" in df.columns:
        before = len(df)
        df = df[df["RegTypeName"] != "Випускник минулих років"].copy()
        print(f"   Dropped {before - len(df):,} 'Випускник минулих років' rows")
    else:
        print(f"   ⚠️  'RegTypeName' column not found — filter skipped for {year}")

    # ── Resolve math score column ──────────────────────────────────────────────
    math_col = MATH_COL_MAP[year]
    if math_col not in df.columns:
        # Fallback: search for any column with "ball100" or "math" in the name
        candidates = [c for c in df.columns
                      if "ball100" in c.lower() or "math" in c.lower()]
        if candidates:
            math_col = candidates[0]
            print(f"   ℹ️  Expected math col '{MATH_COL_MAP[year]}' not found, "
                  f"using fallback: '{math_col}'")
        else:
            print(f"   ❌ No math score column found for {year} — skipping year.")
            continue

    # ── Resolve PTRegName column ───────────────────────────────────────────────
    ptreg_col = PTREG_COL_MAP[year]
    if ptreg_col not in df.columns:
        candidates = [c for c in df.columns if "ptreg" in c.lower()]
        if candidates:
            ptreg_col = candidates[0]
            print(f"   ℹ️  Expected PTReg col '{PTREG_COL_MAP[year]}' not found, "
                  f"using fallback: '{ptreg_col}'")
        else:
            print(f"   ⚠️  No PTRegName column found for {year} — will be NaN.")
            df["PTRegName"] = pd.NA
            ptreg_col = "PTRegName"

    # ── Select and rename year-specific columns ────────────────────────────────
    stable_present = [c for c in STABLE_COLS if c in df.columns]
    missing_stable = [c for c in STABLE_COLS if c not in df.columns]
    if missing_stable:
        print(f"   ⚠️  Missing stable columns for {year}: {missing_stable}")

    cols_to_keep = stable_present + [ptreg_col, math_col]
    df = df[cols_to_keep].copy()

    df = df.rename(columns={
        ptreg_col: "PTRegName",
        math_col:  "MathBall100",
    })

    # ── Normalise math score (some years use comma as decimal separator) ───────
    df["MathBall100"] = (
        df["MathBall100"]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )

    # ── Add year indicator ─────────────────────────────────────────────────────
    df["Year"] = year

    print(f"   ✅ {len(df):,} rows retained")
    frames.append(df)

# ── Concatenate all years ──────────────────────────────────────────────────────
if not frames:
    raise RuntimeError("No data loaded — check file paths and column names.")

merged = pd.concat(frames, ignore_index=True)

# Enforce consistent column order
FINAL_COLS = ["outid", "RegName", "RegTypeName", "PTRegName",
              "SexTypeName", "TerTypeName", "MathBall100",
              "Birth", "TestDate", "Year"]
merged = merged[[c for c in FINAL_COLS if c in merged.columns]]

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"\n📊 Merged dataset shape: {merged.shape}")
print(merged.dtypes)
print(f"\nRows per year:\n{merged['Year'].value_counts().sort_index()}")
print(f"\nMathBall100 nulls: {merged['MathBall100'].isna().sum():,} "
      f"({merged['MathBall100'].isna().mean():.1%})")
print(merged.head())

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = "data/nmt_merged2021-2025.csv"
merged.to_csv(out_path, index=False, encoding="utf-8")
print(f"\n💾 Saved → {out_path}")


📂 Loading 2021...
   Raw shape: (389309, 147)
   Dropped 51,009 'Випускник минулих років' rows
   ⚠️  Missing stable columns for 2021: ['TestDate']
   ✅ 338,300 rows retained

📂 Loading 2022...
   Raw shape: (234103, 29)
   Dropped 36,129 'Випускник минулих років' rows
   ✅ 197,974 rows retained

📂 Loading 2023...


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd1 in position 4072: invalid continuation byte

In [ ]:
import pandas as pd
import os

# ── Column mappings per year ───────────────────────────────────────────────────────
MATH_COL_MAP = {
    2021: "MathBall100",
    2022: "Block3Ball100",
    2023: "MathBlockBall100",
    2024: "MathBlockBall100",
    2025: "MathBlockBall100"
}

PTREG_COL_MAP = {
    2021: "MathPTRegName",
    2022: "PTRegName",
    2023: "PTRegName",
    2024: "PTRegName",
    2025: "PTRegName"
}

FILE_MAP = {
    2021: "data/Odata2021File.csv",
    2022: "data/Odata2022File.csv",
    2023: "data/Odata2023File.csv",
    2024: "data/Odata2024File.csv",
    2025: "data/Odata2025File.csv"
}

STABLE_COLS = ["outid", "RegName", "RegTypeName",
               "SexTypeName", "TerTypeName", "Birth", "TestDate"]

# Encodings to try in order
ENCODINGS_TO_TRY = ["utf-8-sig", "windows-1251", "utf-8", "latin-1"]


def read_csv_any_encoding(filepath: str, **kwargs) -> pd.DataFrame:
    """Try multiple encodings until one works."""
    for enc in ENCODINGS_TO_TRY:
        try:
            df = pd.read_csv(filepath, encoding=enc, **kwargs)
            print(f"   ✅ Encoding: '{enc}'")
            return df
        except (UnicodeDecodeError, UnicodeError):
            print(f"   ✗  Encoding '{enc}' failed, trying next...")
    raise UnicodeDecodeError(
        f"Could not decode {filepath} with any of: {ENCODINGS_TO_TRY}"
    )


READ_KWARGS = dict(
    sep=";",
    quotechar='"',
    engine="python",
    on_bad_lines="skip",
    encoding_errors="replace"
)

# ── Load, filter, rename, collect ─────────────────────────────────────────────
frames = []

for year, filepath in FILE_MAP.items():
    if not os.path.exists(filepath):
        print(f"⚠️  File not found, skipping: {filepath}")
        continue

    print(f"\n📂 Loading {year}...")
    df = read_csv_any_encoding(filepath, **READ_KWARGS)
    print(f"   Raw shape: {df.shape}")

    # ── Filter out previous-year graduates ────────────────────────────────────
    if "RegTypeName" in df.columns:
        before = len(df)
        df = df[df["RegTypeName"] != "Випускник минулих років"].copy()
        print(f"   Dropped {before - len(df):,} 'Випускник минулих років' rows")
    else:
        print(f"   ⚠️  'RegTypeName' not found — filter skipped for {year}")

    # ── Resolve math score column ──────────────────────────────────────────────
    math_col = MATH_COL_MAP[year]
    if math_col not in df.columns:
        candidates = [c for c in df.columns
                      if "ball100" in c.lower() or "math" in c.lower()]
        if candidates:
            math_col = candidates[0]
            print(f"   ℹ️  Math col fallback: '{math_col}'")
        else:
            print(f"   ❌ No math score column found for {year} — skipping year.")
            continue

    # ── Resolve PTRegName column ───────────────────────────────────────────────
    ptreg_col = PTREG_COL_MAP[year]
    if ptreg_col not in df.columns:
        candidates = [c for c in df.columns if "ptreg" in c.lower()]
        if candidates:
            ptreg_col = candidates[0]
            print(f"   ℹ️  PTReg col fallback: '{ptreg_col}'")
        else:
            print(f"   ⚠️  No PTRegName column found for {year} — will be NaN.")
            df["PTRegName"] = pd.NA
            ptreg_col = "PTRegName"

    # ── Select and rename ──────────────────────────────────────────────────────
    stable_present = [c for c in STABLE_COLS if c in df.columns]
    missing_stable = [c for c in STABLE_COLS if c not in df.columns]
    if missing_stable:
        print(f"   ⚠️  Missing stable columns: {missing_stable}")

    df = df[stable_present + [ptreg_col, math_col]].copy()
    df = df.rename(columns={ptreg_col: "PTRegName", math_col: "MathBall100"})

    # ── Normalise math score ───────────────────────────────────────────────────
    df["MathBall100"] = (
        df["MathBall100"]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )

    df["Year"] = year
    print(f"   ✅ {len(df):,} rows retained")
    frames.append(df)

# ── Concatenate ────────────────────────────────────────────────────────────────
if not frames:
    raise RuntimeError("No data loaded — check file paths and column names.")

merged = pd.concat(frames, ignore_index=True)

FINAL_COLS = ["outid", "RegName", "RegTypeName", "PTRegName",
              "SexTypeName", "TerTypeName", "MathBall100",
              "Birth", "TestDate", "Year"]
merged = merged[[c for c in FINAL_COLS if c in merged.columns]]

print(f"\n📊 Merged shape: {merged.shape}")
print(f"\nRows per year:\n{merged['Year'].value_counts().sort_index()}")
print(f"MathBall100 nulls: {merged['MathBall100'].isna().sum():,} "
      f"({merged['MathBall100'].isna().mean():.1%})")

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = "data/nmt_merged2021-2025.csv"
merged.to_csv(out_path, index=False, encoding="utf-8")
print(f"\n💾 Saved → {out_path}")


📂 Loading 2021...


In [2]:
import chardet

with open("data/Odata2025File.csv", "rb") as f:
    raw = f.read(100_000)  # read first 100kb, enough to detect

result = chardet.detect(raw)
print(result)

{'encoding': 'UTF-8-SIG', 'confidence': 1.0, 'language': 'uk', 'mime_type': 'text/plain'}


In [4]:
with open("data/Odata2023File.csv", "rb") as f:
    raw = f.read()

# Find the first byte that breaks utf-8
for i, byte in enumerate(raw):
    try:
        raw[i:i+4].decode("utf-8")
    except UnicodeDecodeError:
        print(f"Bad byte at position {i}: 0x{raw[i]:02x}")
        print(f"Context: {repr(raw[max(0,i-20):i+20])}")
        break

Bad byte at position 1: 0xbb
Context: b'\xef\xbb\xbf"outid";"Birth";"S'


In [8]:
df = pd.read_csv(
    "data/Odata2023File.csv",
    sep=";",
    quotechar='"',
    engine="python",
    on_bad_lines="skip",
    encoding="utf-8-sig"   # 👈 handles BOM (\xef\xbb\xbf) correctly
)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd1 in position 4072: invalid continuation byte